# Used Vehicle Price Intelligence
### Predicting resale value, investigating pricing signals and testing model reliability

**Research case study · Python / Pandas / Scikit-learn · Sonkhe Shongwe**

**Research question:** How accurately can a used vehicle's selling price be estimated from its recorded characteristics, how much of that accuracy depends on a known reference price, and where do our predictions break down?

**Executive finding:** A **Random forest** model selected from development-only five-fold cross-validation reaches **R² 0.934** and **MAE 0.70 lakh INR** on 60 protected test observations. Yet removing its reference-price feature raises development CV MAE by a factor of **2.4**. High-value errors are much larger than the pooled average. Our central story is about **reliability**, not a leaderboard.

> This dataset is historical and small. The project is not a current market estimator, and none of the predictive relationships are causal effects. Price units are interpreted from matching CarDekho source descriptions; the CSV itself does not record a currency.

![Used Vehicle Price Intelligence: project executive overview](assets/00_portfolio_cover.png)

## 1 · Decision context and analytical roadmap

A buyer or seller may want an independent value estimate. A successful analytical system must do more than produce a number: it must explain what information is necessary to support the prediction and under what conditions it may be wrong.

This investigation proceeds from **dataset reality → market segmentation → price relationships → honest model selection → error analysis → feature-dependence experiment → generalisation stress test → limitations**.

**Questions we will answer**

1. Are the rows all cars, and does vehicle category alter pricing comparisons?
2. Can a model beat a simple median-price baseline under consistent cross-validation?
3. How reliably does the selected algorithm perform on unseen observations?
4. How much performance disappears if the reference price is not provided?
5. Which vehicle segments and unfamiliar model names challenge the prediction?

The published figures are generated from the analysis itself. The notebook is reproducible from the included CSV and source files.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display, Image, Markdown
from src.analysis import load_data, main_analysis, point_metrics
from src.visuals import generate_all_figures
ROOT = Path.cwd()
print('Running from:', ROOT.resolve())
print('Dataset available:', (ROOT / 'car_data.csv').exists())

Running from: /mnt/data/car_price_rebuild/car-price-prediction
Dataset available: True


## 2 · Dataset provenance and record-level audit

**Source copy:** [Kaggle / CarDekho vehicle dataset](https://www.kaggle.com/datasets/nehalbirla/vehicle-dataset-from-cardekho). The original CSV has 301 rows, nine columns, and no missing values. Prices are commonly documented in **lakhs of Indian rupees**, but the file itself provides neither a currency field nor a transaction date.

`Selling_Price` is the predicted target. `Present_Price` is described as a current showroom/reference price in copies of the source documentation, but we cannot establish its observation date from the CSV. `Car_Name` is a vehicle model name, `Year` is a manufacture/model year, and `Driven_kms` is recorded mileage.

**Critical decisions:** drop only exact duplicate rows; retain the extreme Land Cruiser; review each model name to classify cars versus motorcycles/scooters. Never infer vehicle class from seller type or target price. Reject unknown model labels rather than silently assigning them to a class.

**No feature derived from `Selling_Price` may enter the training pipeline.**

In [2]:
df, audit = load_data(ROOT / 'car_data.csv')
display(pd.DataFrame([audit]).T.rename(columns={0:'Verified value'}))
display(df[['Car_Name','Vehicle_Type','Year','Selling_Price','Present_Price','Driven_kms','Fuel_Type','Selling_type']].head(8))

,Verified value
source_rows,301
missing_cells,0
exact_duplicates,2
source_model_names,98
analytical_rows,299
cars,198
two_wheelers,101
model_names,98
vehicle_year_min,2003
vehicle_year_max,2018


,Car_Name,Vehicle_Type,Year,Selling_Price,Present_Price,Driven_kms,Fuel_Type,Selling_type
0,ritz,Car,2014,3.35,5.59,27000,Petrol,Dealer
1,sx4,Car,2013,4.75,9.54,43000,Diesel,Dealer
2,ciaz,Car,2017,7.25,9.85,6900,Petrol,Dealer
3,wagon r,Car,2011,2.85,4.15,5200,Petrol,Dealer
4,swift,Car,2014,4.60,6.87,42450,Diesel,Dealer
5,vitara brezza,Car,2018,9.25,9.83,2071,Diesel,Dealer
6,ciaz,Car,2015,6.75,8.12,18796,Petrol,Dealer
7,s cross,Car,2015,6.50,8.61,33429,Diesel,Dealer


### Audit interpretation

After dropping **2 exact duplicate rows**, the analytical sample contains **299** unique records: **198 cars** and **101 two-wheelers**. The source file has **no null cells**, yet its market composition is unbalanced. All two-wheeler records are listed as individual sellers, while only **2.5%** of car records share that channel.

This makes apparent seller-type premiums vulnerable to **confounding**: seller type partly acts as a proxy for which market segment is represented. We will not interpret its correlation as a causal markup.

`Year` ranges from **2003 to 2018**. Without a verified valuation reference date, we cannot convert it to a precise age. The highest reference-price observation is the **Land Cruiser (92.6 lakh)** and remains in our primary analysis.

![Car and two-wheeler distributions and seller channels](assets/01_vehicle_mix.png)

**Interpretation:** absolute selling prices differ substantially by vehicle category, and seller-channel composition is uneven. Using a single pooled model may be reasonable as an experiment, but any explanation that ignores this structure would be misleading.

In [3]:
display(df.groupby('Vehicle_Type').agg(
    count=('Car_Name','size'), median_selling_price=('Selling_Price','median'),
    median_reference_price=('Present_Price','median'),
    median_mileage=('Driven_kms','median')
).round(3))
display(pd.crosstab(df.Vehicle_Type,df.Selling_type,margins=True))

,count,median_selling_price,median_reference_price,median_mileage
Vehicle_Type,,,,
Car,198,5.25,8.40,39742.5
Two-wheeler,101,0.50,0.84,18000.0


Selling_type,Dealer,Individual,All
Vehicle_Type,,,
Car,193,5,198
Two-wheeler,0,101,101
All,193,106,299


## 3 · What is associated with resale prices?

A reference-price field is unusually informative: its value can already approximate a vehicle's price segment. We first examine the observed association without interpreting it as a causal effect.

A tempting proxy for depreciation is the **resale-to-reference-price ratio**. We may calculate it for descriptive reporting, but because its numerator is the target, it is **forbidden as a predictive feature**. It is also not true age-adjusted depreciation from the original purchase price because that purchase date and price are unknown.

A high-priced observation should be investigated, not automatically discarded. The Land Cruiser illustrates exactly why retaining challenging cases matters for a credibility-focused model.

![Reference price versus recorded selling price](assets/02_price_relationship.png)

In [4]:
df['Descriptive_Retention_Ratio'] = df['Selling_Price'] / df['Present_Price']
display(df.groupby('Vehicle_Type')['Descriptive_Retention_Ratio'].agg(['count','median','min','max']).round(3))
print('Ratios above 1:',int((df.Descriptive_Retention_Ratio>1).sum()),' (inspect definitions before calling these errors)')
df = df.drop(columns='Descriptive_Retention_Ratio')

,count,median,min,max
Vehicle_Type,,,,
Car,198,0.654,0.105,0.989
Two-wheeler,101,0.625,0.133,0.947


Ratios above 1: 0  (inspect definitions before calling these errors)


## 4 · Design a fair predictive experiment

**Prediction inputs:** `Year`, `Driven_kms`, `Owner`, `Present_Price`, `Fuel_Type`, `Selling_type`, `Transmission`, and an explicitly reviewed `Vehicle_Type` label. The model name is deliberately omitted from the main feature matrix to avoid memorising a few frequent names in this small dataset.

**Protected evaluation protocol:**

1. Remove only exact duplicates **before** splitting.
2. Stratify a random 80/20 development/test split by vehicle category (seed 42).
3. Fit imputation, numeric scaling and categorical one-hot encoding *inside* each pipeline and CV fold.
4. Use shuffled five-fold development-set CV with the same folds for every model.
5. Select by **lowest mean CV MAE**, not by the protected test's R².
6. Fit the selected candidate to the full development partition.
7. Evaluate the held-out test set **once**, then use it only for diagnostic analysis rather than tuning.

**Model candidates:** median-only benchmark, Ridge, Random Forest, Extra Trees and Gradient Boosting. These are intentionally fixed, limited-capacity candidate configurations, not a hidden hyperparameter search. Pipelines prevent training folds from seeing transformations fitted on held-out rows.

The 80/20 split is stratified only by vehicle class, not target-derived price bands. A separate model-name-disjoint split addresses one type of distribution shift later.

In [5]:
# Single reproducible execution creates the numeric reports and all eight figures.
results = main_analysis(ROOT/'car_data.csv', ROOT/'reports')
generated = generate_all_figures(results, ROOT/'assets')
print('All publication figures generated:',len(generated))
print('Selected candidate:',results['summary']['champion'])
display(results['table'].round(3))

All publication figures generated: 8
Selected candidate: Random forest


,Model,CV_MAE,CV_MAE_SD,CV_RMSE,CV_R2
0,Random forest,0.684,0.238,1.594,0.881
1,Extra Trees,0.727,0.199,1.609,0.877
2,Gradient boosting,0.907,0.187,1.794,0.850
3,Ridge regression,1.219,0.178,1.775,0.839
4,Median baseline,3.291,0.647,4.734,-0.085


## 5 · Model selection by cross-validation

The development-only comparison selected **Random forest** with mean fold MAE **0.684 lakh INR**, compared with **3.291 lakh** for the median-only baseline. Extra Trees was close, at **0.727 lakh**.

This differs from the original notebook, where Extra Trees was chosen from a single test comparison after dropping the Land Cruiser. The new result reflects **a different dataset and, crucially, a different selection protocol**. There is no defensible reason to preserve the original winner for aesthetic purposes.

CV error bars show **fold-to-fold variation**, not confidence intervals for production error.

![Five-fold CV model rankings](assets/03_cross_validation.png)

## 6 · Protected test-set performance

After model selection, we evaluate the chosen model on **60 previously held-out rows**, including **40 cars** and **20 two-wheelers**.

| Metric | Protected test result |
|:--|--:|
| R² | **0.934** |
| MAE | **0.702 lakh INR** |
| RMSE | **1.495 lakh INR** |
| Median absolute error | 0.346 lakh INR |
| 90th-percentile absolute error | 1.293 lakh INR |

**Interpretation:** Predictions track many held-out prices well in this sample. But the protected set contains just 60 rows, the target spans dramatically different value ranges and the model relies on an unusually informative reference price. We cannot infer current-market accuracy or operational reliability from these metrics alone.

In [6]:
display(results['test_rows'].sort_values('Absolute_Error',ascending=False)
    [['Car_Name','Vehicle_Type','Selling_Price','Predicted','Absolute_Error']].head(8).round(3))

,Car_Name,Vehicle_Type,Selling_Price,Predicted,Absolute_Error
63,fortuner,Car,33.00,23.585,9.415
58,fortuner,Car,19.99,23.525,3.535
50,fortuner,Car,23.00,20.132,2.868
92,corolla altis,Car,4.00,6.374,2.374
52,fortuner,Car,16.00,17.485,1.485
258,city,Car,9.15,10.518,1.368
282,brio,Car,3.50,4.785,1.285
21,sx4,Car,4.40,3.121,1.279


![Actual versus predicted prices on protected test set](assets/04_holdout_predictions.png)

## 7 · Where do the errors accumulate?

We examine **price-band** and **vehicle-class** diagnostics after the model is fixed. The target-based bands (under 2, 2–10 and over 10 lakh) are used **for evaluation only**, never training.

An important subtlety: an MAE of 1 lakh on a motorcycle and 1 lakh on an expensive SUV have very different relative meaning. The plotted absolute errors identify operational exposures, not a perfectly controlled comparison across vehicle types.

With just **seven high-priced test records**, the premium-segment result is informative but extremely uncertain. It should trigger caution and further validation, not marketing claims about a predictable luxury vehicle market.

In [7]:
display(results['segments'][['Dimension','Group','Count','MAE','RMSE','MedAE','R2']].round(3))

,Dimension,Group,Count,MAE,RMSE,MedAE,R2
0,Vehicle_Type,Car,40,0.987,1.826,0.593,0.908
1,Vehicle_Type,Two-wheeler,20,0.131,0.192,0.069,0.856
2,Price_Segment,2–10L,33,0.585,0.757,0.525,0.789
3,Price_Segment,Over 10L,7,2.885,4.044,1.485,0.720
4,Price_Segment,Under 2L,20,0.131,0.192,0.069,0.856


![Subgroup error breakdown](assets/05_segment_errors.png)

## 8 · Controlled price-anchor ablation

How much of our model's apparent competence comes from being told a trustworthy current reference price?

For the **same selected algorithm, 239 development rows and five CV folds**, we compare the full predictors with a version missing `Present_Price`.

- With a price anchor: **CV MAE 0.684 lakh**.
- Without an anchor: **CV MAE 1.617 lakh**.
- Error increases by **2.4×**.

**Interpretation:** The feature is highly useful, but that can also limit the practical value of the model. If a trustworthy showroom/reference price is unavailable, the reported full-model accuracy does not apply. The ablation is a development-set sensitivity experiment; it is *not* another final model-selection round on test observations.

In [8]:
display(results['ablation'].round(3))

,Experiment,CV_MAE,CV_RMSE,CV_R2
0,With reference price,0.684,1.594,0.881
1,Without reference price,1.617,2.937,0.592


![Reference price ablation](assets/06_reference_price_ablation.png)

## 9 · Stress-test unfamiliar model names and explain predictor dependence

**Unseen model names.** A separate `GroupShuffleSplit` reserves **69 rows from 20 model names** completely absent from the stress model's training sample. The held-out R² is **0.629**. This indicates sensitivity to the group distribution but does not isolate a causal effect of unfamiliar names: the training/test samples and target variance differ from the primary split. Crucially, we do not pass `Car_Name` into the estimator.

**Expensive-record sensitivity.** Our Land Cruiser is kept in the main analysis. A development-fold calculation that excludes it can change the CV metric, but it simultaneously changes fold composition. We report this as diagnostic information rather than proof that the record alone causes an observed accuracy difference.

**Permutation importance.** After the protected test assessment, we permute each source predictor and measure the mean increase in absolute error. `Present_Price` dominates. This captures predictive reliance, **not independent economic causality**; correlated predictors may share importance.

In [9]:
print('Unseen-name stress test:', results['summary']['unseen_model_stress'])
print('Extreme-record development-only sensitivity:',results['summary']['outlier_sensitivity'])
display(results['imp'].round(3))

Unseen-name stress test: {'count': 69, 'unique_unseen_names': 20, 'metrics': {'MAE': 0.5054155662909284, 'RMSE': 1.2464907729296157, 'MedAE': 0.17334457671957848, 'R2': 0.6293100633517008, 'P90_AE': 0.9481944929453263}}
Extreme-record development-only sensitivity: {'largest_reference_price_record_in_development': True, 'development_CV_MAE_without_record': 0.6108941022482796, 'development_CV_MAE_with_record': 0.6844227442710269}


,Feature,MAE_Increase,Repeat_SD
3,Present_Price,4.568,0.375
0,Year,0.665,0.072
1,Driven_kms,0.106,0.053
4,Fuel_Type,0.038,0.015
5,Selling_type,0.037,0.007
6,Transmission,0.007,0.005
7,Vehicle_Type,0.005,0.002
2,Owner,0.000,0.000


![Robustness and permutation feature importance](assets/07_robustness_and_importance.png)

## 10 · Conclusions, use boundaries and next questions

**The major conclusion is about prediction reliability:** strong pooled holdout performance is possible on this dataset, yet it depends heavily on the known reference-price feature and hides substantially larger absolute errors in expensive vehicles.

### What this study supports

- A reproducible modelling comparison using development-only CV and one protected holdout.
- Evidence that a current reference-price feature is exceptionally predictive *in this dataset*.
- Evidence of distinct vehicle markets and severe seller-type confounding.
- A clear warning that a single high R² is insufficient without segment diagnostics and separate generalisation tests.

### What it does **not** support

- A current-2026 market quotation, dealer margin analysis, or causal depreciation model.
- A precise vehicle-age adjustment, since no transaction/reference year is provided.
- Guaranteed accuracy for premium vehicles or unfamiliar market conditions.
- Independent importance of seller type, given near-complete seller/vehicle-class association.

**Better next step for a real deployment:** secure new representative time-stamped vehicle data containing trim, condition, location and actual transactions. We intentionally do **not** introduce any of those additional data sources into this project.

### Reproducibility confirmation

All eight figures and numeric CSV reports were generated from the included `car_data.csv` using `run_analysis.py`. The source CSV is unmodified and the model output was not edited to preserve the old Extra Trees result. See the accompanying methodology and model card for full assumptions and audit notes.

*Research credit:* [Kaggle CarDekho vehicle dataset](https://www.kaggle.com/datasets/nehalbirla/vehicle-dataset-from-cardekho).